# - GPT에게 기억력을 심어주자, Memory
## 1. 랭체인 내장 메모리
### ChatPromptTemplate 복습
### 필요 패키지 설치
- 아래 코드 셀을 실행하여 필요한 패키지를 설치.

In [ ]:
# %pip install langchain langchain-openai openai

In [ ]:
%pip show langchain
%pip show langchain-openai
%pip show openai
%pip show langchain-community

In [3]:
from dotenv import load_dotenv

load_dotenv()

True

#### 체인 정의

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import  ChatOpenAI

prompt = ChatPromptTemplate([
    ('system', '당신은 강아지 안구 건강 관련 조언을 해주는 AI 챗봇입니다.'),  # 역할
    ('human', '안녕하세요! 저희 집 강아지가 눈을 자주 긁어요'),    # langchain에서는 human으로 정의
    ('ai', '그렇군요. 강아지는 몇 살인가요?'),
    ('human', '{user_input}')
])

model = ChatOpenAI(model='gpt-5-nano')
output_parser = StrOutputParser()

chain = prompt | model | output_parser
chain.invoke({'user_input':'3살이에요.'})  # 다음 질문은 대답 겸 질문

# 대화의 맥락을 기억하지 못하는 단점이 존재함

'강아지가 눈을 자주 긁는 건 여러 원인이 있을 수 있어요. 3살이라도 빠르게 정확한 진단이 필요할 수 있습니다. 아래 정보를 알려주시면 원인 추정과 다음 단계 계획을 더 잘 도와드릴 수 있어요.\n\n먼저 지금 체크해볼 점\n- 눈이 한쪽인가요, 아니면 양쪽인가요?\n- 눈 주위가 빨개졌나요? 눈 자체가 붉어 보이나요?\n- 눈에서 분비물(투명한 눈물 vs 노랗거나 초록색 농)이 나나요?\n- 강아지가 눈을 비비거나 눈을 거의 감고 있나요? 통증 징후가 보이나요? 빛에 민감해 보이나요?\n- 시작 시점은 언제부터였나요? 특정 상황(꽃가루가 많은 날, 새로 산 장난감, 먼지 등)과 관련이 있나요?\n- 한동안 외출 후나 실내 공기질이 안 좋던 날이 있었나요?\n- 현재 다른 증상(발열, 식욕 부진, 기운 없음)이 있나요?\n\n가능한 원인들\n- 알레르기나 자극에 의한 결막염\n- 건성안(건조증) 또는 눈물샘 기능 저하\n- 각막 손상이나 이물질로 인한 통증\n- 세균/바이러스성 감염\n- 이물질이나 눈꺼풀 모양 이상(예: 눈꺼풀 안쪽으로 말려드는 엔트로피 등)\n- 특정 품종에서 잘 생길 수 있는 눈 주변 문제(구형체 큰 눈, 주름이 많은 품종에서 자주 생김)\n\n지금 바로 할 수 있는 안전한 조치\n- 손으로 눈을 비비지 못하게 주의시키되, 만약 억지로 비비는 경우라면 바람직하지 않아요.\n- 눈 주위를 부드러운 천이나 면으로 더럽지 않게 닦아낼 때는 따뜻한 물에 살짝 적신 뒤 바깥쪽에서 안쪽으로 닦아주세요. 내부로 물이 들어가서는 안 돼요.\n- 인간용 눈물방울이나 약은 절대 사용하지 마세요. 수의사가 처방한 동물용 눈 약이나 인공눈물(동물용으로 안전한 것)만 사용하도록 권장합니다.\n- 이물질이 눈 속으로 들어갔다고 의심되면 억지로 제거하려고 하지 말고, 전문의와 상의하세요.\n- 가능하면 외부 환경(먼지, 담배 연기, 화학세제 냄새 등)을 줄이고, 알레르기 시즌이라면 창문을 닫아 공기를 차단해 주세요.\n\n언제 수의사 상담이 급한가요(응급 신호)\

### 1.1 ChatMessageHistory

#### langchain-community 설치

In [ ]:
# %pip install -U langchain-community

#### ChatMessageHistory 객체 생성
- 대화 내역을 저장하고 관리하기 위한 객체.

In [ ]:
from langchain_community.chat_message_histories import ChatMessageHistory

history= ChatMessageHistory() 
print(history.messages)  # 히스토리 관리 및 저장 [대화의 내역]


[]


#### 대화 내역 추가
- 이번에는 생성한 history 객체에 대화 내역을 추가.
- 주의할 점은 프롬프트를 작성할 때와 마찬가지로 누구의 대화 내용을 의미하는지를 명시해야 함.

In [ ]:
history.add_user_message('인녕 내 이름은 철수야.')  # 사용자의 질문
history.add_ai_message('안녕하세요, 철수님! 반갑습니다.') 

for message in history.messages:
    print(message.content)
    

인녕 내 이름은 철수야.
안녕하세요, 철수님! 반갑습니다.


### 1.2 MessagesPlaceholder
#### 프롬프트 정의
- 리스트의 요소에 MessagesPlaceholder 객체 자체를 추가하여 프롬프트를 정의.

In [11]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_openai import ChatOpenAI

# 기존 방식
# prompt = ChatPromptTemplate([
#     ("system", "당신은 강아지 안구 건강 관련 조언을 해주는 AI 챗봇입니다."),
#     ("human", "안녕하세요! 저희 집 강아지가 눈을 자주 긁어요"),
#     ...
# ])

# → → 대화의 맥락을 기억하지 못하는 단점이 존재함

# 새로운 방식
# 리스트의 요소에 MessagesPlaceholder 객체 자체를 추가하여 프롬프트를 정의
prompt = ChatPromptTemplate([
    ("system", "당신은 강아지 안구 건강 관련 조언을 해주는 AI 챗봇입니다."),
    MessagesPlaceholder(variable_name='chat_history'),    # 대화 맥락은 무엇인지를 나타내기 위해 variable_name을 지정
    ('human', '{user_input}')                             # 마지막은 사용자의 질문
])



#### 대화 내역 저장소 생성
- 프롬프트에서 사용할 '이전 대화들'을 저장한 대화 내역 저장소를 생성.

In [ ]:
from langchain_community.chat_message_histories import ChatMessageHistory

# 새롭게 추가한 질문과 모델의 답변을 new_history에 추가
new_history = ChatMessageHistory()  

new_history.add_user_message('안녕하세요. 저희 집 고양이의 이름은 치즈이고, 물고기를 좋아합니다.')
new_history.add_ai_message('안녕하세요. 물고기를 좋아하는 고양이 치즈를 데리고 계시군요!')

print(len(new_history.messages))


2


####	구성 요소 정의와 체인 생성
- 체인을 구성할 모델과 출력 파서를 정의하고 체인을 연결.

In [13]:

from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model='gpt-5-nano')
output_parser = StrOutputParser()

chain = prompt | model | output_parser

question = '그리고 츄르도 좋아합니다.'
answer = chain.invoke({
    'chat_history': new_history.messages,   # MessagesPlaceholder:히스토리 데이터를 가져오는 기능을 가지고 있으면서 위치만 지정해 주면 메시지를 호출하면서 동기화
    'user_input': question                  # 질문의 내용
    })

print(answer)


치즈가 츄르를 좋아하고 물고기도 좋아하는 모습이 귀엽네요!

참고로 저는 강아지의 안구 건강에 특화되어 있지만, 고양이 눈 건강에 대한 기본 관리 팁도 간단히 정리해 드릴게요. 아래를 참고해 보시고, 눈 문제가 의심되면 수의사와 상담해 주세요.

고양이 눈 건강 관리 팁
- 눈 상태 매일 확인하기: 눈이 붉어지거나, 비정상적으로 눈물이 많아지거나, 노란/초록색 분비물, 눈꺼풀 부종, 눈이 잘 안 떠지는 등의 이상 증상이 있는지 관찰해 보세요.
- 필요할 때만 눈 닦기: 분비물이 있을 때만 damp한 면으로 부드럽게 닦아 주세요. 한 면으로 여러 번 닦지 말고, 깨끗한 면으로 바꿔 가며 닦으세요.
- 자극물 피하기: 연기, 먼지, 강한 바람 등 눈에 자극이 되는 환경을 피하고 환기와 청결을 유지해 주세요.
- 눈 주위 관리: 눈 주위 털이 길면 눈에 자극이 될 수 있어 정리해 주면 좋습니다. 눈에 직접 닿는 부위를 깨끗하게 유지하세요.
- 간식 관리: 츄르는 일반적으로 안전하지만 과다 급여로 체중이 증가하면 건강에 좋지 않을 수 있어 균형 있게 주세요. 눈 건강 자체에 직접적인 효과는 크지 않습니다.
- 약물은 수의사 처방 없이 사용하지 않기: 눈에 바르는 약이나 안약은 반드시 수의사의 처방에 따라 사용하세요.
- 이상 징후가 보이면 바로 수의사 방문: 지속적인 발적, 다량의 분비물, 눈에 통증이나 긁는 행동, 시력 저하 의심 등이 있으면 병원을 찾아 진료 받으세요.

궁금한 점이나 치즈의 최근 눈 상태에 대해 알려주시면, 상황에 맞춰 더 구체적인 조언을 드리겠습니다. 예를 들어 최근에 눈이 자주 발적이나 분비물이 생기나요? 어떤 증상이 있는지 알려주실 수 있을까요?


#### 이전 대화 맥락 저장소에 추가

In [17]:
new_history.add_user_message(question)
new_history.add_ai_message(answer)

#### new_history 구성 확인

In [18]:
for message in new_history.messages:
    speaker = '사용자' if 'human' in str(type(message)) else 'AI'
    print(f'{speaker}: {message.content}')

사용자: 안녕하세요. 저희 집 고양이의 이름은 치즈이고, 물고기를 좋아합니다.
AI: 안녕하세요. 물고기를 좋아하는 고양이 치즈를 데리고 계시군요!
사용자: 그리고 츄르도 좋아합니다.
AI: 치즈가 츄르를 좋아하고 물고기도 좋아하는 모습이 귀엽네요!

참고로 저는 강아지의 안구 건강에 특화되어 있지만, 고양이 눈 건강에 대한 기본 관리 팁도 간단히 정리해 드릴게요. 아래를 참고해 보시고, 눈 문제가 의심되면 수의사와 상담해 주세요.

고양이 눈 건강 관리 팁
- 눈 상태 매일 확인하기: 눈이 붉어지거나, 비정상적으로 눈물이 많아지거나, 노란/초록색 분비물, 눈꺼풀 부종, 눈이 잘 안 떠지는 등의 이상 증상이 있는지 관찰해 보세요.
- 필요할 때만 눈 닦기: 분비물이 있을 때만 damp한 면으로 부드럽게 닦아 주세요. 한 면으로 여러 번 닦지 말고, 깨끗한 면으로 바꿔 가며 닦으세요.
- 자극물 피하기: 연기, 먼지, 강한 바람 등 눈에 자극이 되는 환경을 피하고 환기와 청결을 유지해 주세요.
- 눈 주위 관리: 눈 주위 털이 길면 눈에 자극이 될 수 있어 정리해 주면 좋습니다. 눈에 직접 닿는 부위를 깨끗하게 유지하세요.
- 간식 관리: 츄르는 일반적으로 안전하지만 과다 급여로 체중이 증가하면 건강에 좋지 않을 수 있어 균형 있게 주세요. 눈 건강 자체에 직접적인 효과는 크지 않습니다.
- 약물은 수의사 처방 없이 사용하지 않기: 눈에 바르는 약이나 안약은 반드시 수의사의 처방에 따라 사용하세요.
- 이상 징후가 보이면 바로 수의사 방문: 지속적인 발적, 다량의 분비물, 눈에 통증이나 긁는 행동, 시력 저하 의심 등이 있으면 병원을 찾아 진료 받으세요.

궁금한 점이나 치즈의 최근 눈 상태에 대해 알려주시면, 상황에 맞춰 더 구체적인 조언을 드리겠습니다. 예를 들어 최근에 눈이 자주 발적이나 분비물이 생기나요? 어떤 증상이 있는지 알려주실 수 있을까요?


#### 새로운 대화를 기억하고 있는지 확인

In [19]:
question = "우리 고양이가 물고기 말고 또 무엇을 좋아한다고 했었죠?"

answer = chain.invoke({
    'chat_history': new_history.messages,
    'user_input': question
})

print(answer)

맞아요. 치즈는 물고기도 좋아하고, 츄르도 좋아한다고 하셨죠. 

혹시 치즈의 눈 건강 관리나 간식 관리에 대해 더 궁금한 점이 있을까요? 최근 눈 상태나 눈 주위 털 관리에 대해 알려주시면 구체적으로 도와드리겠습니다.


## 2. Runnable 활용
### 2.1 RunnableWithMessageHistory 기본
#### 대화 내역 저장소 생성
- get_chat_history 함수를 통해 객체를 반환 받도록 함.

In [20]:
from langchain_community.chat_message_histories import ChatMessageHistory

auto_history = ChatMessageHistory()

def get_chat_history():
    return auto_history


#### 프롬프트 정의

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_openai import ChatOpenAI

# ChatPromptTemplate 대화의 맥락을 기억하는 프롬프트
prompt = ChatPromptTemplate([
    'system', '당신은 친절한 한국어 비서입니다.',
    MessagesPlaceholder(variable_name='history'),
    ('human', '{question}')   # 사용자 질문으로 GPT와의 통신이 시작
])   


#### 구성 요소 정의와 체인 생성

In [32]:
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model='gpt-5-nano')
output_parser = StrOutputParser()

chain = prompt | model | output_parser

#### RunnableWithMessageHistory 객체 정의
- 첫 번째 인자는 반드시 Runnable 객체여야 함.

In [33]:
from langchain_core.runnables.history import RunnableWithMessageHistory

chain_with_message = RunnableWithMessageHistory(
    chain,
    get_chat_history,                # 대화 내역 저장소를 가져오는 함수
    input_messages_key='question',   # 입력 이름 
    history_messages_key='history'   # 기록 이름 MessagePlaceHolder에 저장한 변수
)

c:\workspaces\ai_agent\src\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3775: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


####	연속된 대화 테스트

In [ ]:
# 첫 번째 대화
question1 = "안녕! 내 직업은 영화 감독이야."

# RunnableWithMessageHistory 객체를 통해 실행, 주고 받은 메시지의 내역을 내부에서 관리

response1 = chain_with_message.invoke(
    {"question": question1},
)

print(f"사용자: {question1}")
print(f"AI: {response1}")

# 두 번째 대화
question2 = "내 직업 기억해?"

response2 = chain_with_message.invoke(
    {"question": question2},
)
print(f"사용자: {question2}")
print(f"AI: {response2}")

사용자: 안녕! 내 직업은 영화 감독이야.
AI: 네, 영화 감독님. 반갑습니다! 지금 바로 도와드릴 수 있어요. 어떤 영역부터 시작할까요? 예시를 드리자면:

- 아이디어 발전: 로그라인 3개, 시놉시스 초안
- 스크립트/구성: 구조 구성, 캐릭터 아크, 대사 다듬기
- 프리프로덕션: 촬영계획, 샷 리스트(테크니컬 샷 포함), 미술/의상 컨셉
- 예산/일정: 예산 추정, 일정표 초안
- 포스트 프로덕션: 편집 방향, 음악/사운드 디자인, 색보정 아이디어
- 팀 소통: 디렉터 노트 정리, 리허설 및 커뮤니케이션 팁

현재 프로젝트의 간단한 정보(장르, 분위기, 기간, 예산 규모 등)나 구체적으로 해결하고 싶은 문제를 알려주시면 바로 맞춤 초안을 만들어 드리겠습니다. 로그라인이 필요하신가요, 아니면 샷 리스트 초안부터 시작할까요?
사용자: 내 직업 기억해?
AI: 네, 기억합니다. 당신은 영화 감독이시죠.

지금 바로 도와드릴 수 있어요. 어떤 영역부터 시작할까요? 예시로는
- 아이디어 발전: 로그라인 3개, 시놉시스 초안
- 스크립트/구성: 구조 구상, 캐릭터 아크, 대사 다듬기
- 프리프로덕션: 촬영계획, 샷 리스트 초안, 미술/의상 컨셉
- 예산/일정: 예산 추정, 일정표 초안
- 포스트 프로덕션: 편집 방향, 음악/사운드, 색보정 아이디어
- 팀 소통: 디렉터 노트 정리, 리허설 및 커뮤니케이션 팁

현재 프로젝트에 대한 간단한 정보(장르, 분위기, 기간, 예산 규모 등)나 해결하고 싶은 구체적 이슈를 알려주시면 바로 맞춤 초안을 드리겠습니다. 로그라인이 필요하신가요, 아니면 샷 리스트부터 시작할까요?


### 2.2 사용자별 대화 관리
#### 대화 내역 저장소 생성

In [35]:
from langchain_community.chat_message_histories import ChatMessageHistory

user_chats = {} # 사용자별 대화를 저장하는 딕셔너리

def get_user_chats(session_id):
    if session_id not in user_chats:
        user_chats[session_id] = ChatMessageHistory()  # 한 번도 대화하지 않은 사용자 

    return user_chats[session_id]   # 기존에 대화한 사용자

#### runnable 객체 구성

In [36]:
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

prompt = ChatPromptTemplate([
    ("system", "당신은 각 상황에 대응 가능한 박학다식한 한국어 비서입니다."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{question}")
])
model = ChatOpenAI(model="gpt-5-nano")
output_parser = StrOutputParser()

chain = prompt | model | output_parser

chain_with_session_id =RunnableWithMessageHistory(
    chain,
    get_user_chats,
    input_messages_key='question',
    history_messages_key='history'
)

c:\workspaces\ai_agent\src\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3775: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


#### 대화 시작

In [ ]:
# 첫 번째 사용자와 대화
question1 = "안녕하세요. 저는 선형 대수가 궁금합니다."

response1 = chain_with_session_id.invoke(
    {"question": question1},
    config={"session_id": "user1"}  # 사용자1, dictionary 형태로 지정
)

print(f"사용자1: {question1}")
print(f"AI: {response1}")

사용자1: 안녕하세요. 저는 선형 대수가 궁금합니다.
AI: 반갑습니다! 선형 대수는 벡터와 행렬을 다루며, 선형 관계와 변환의 구조를 이해하는 학문입니다. 시작하기에 앞서 몇 가지 핵심 주제를 간단히 소개하고, 원하시는 방향으로 맞춰 드릴게요.

주요 개념 정리
- 벡터와 벡터 공간: 벡터의 합성, 스칼라 곱, 기저와 차원
- 행렬과 연산: 덧셈/스칼라곱, 행렬 곱, 역행렬, 행렬식
- 선형 시스템: 연립선형방정식의 해(해의 존재성과 유일성), 랭크
- 고유값/고유벡터와 대각화: 선형 변환의 구조를 단순화
- 선형 변환과 기하적 해석: 벡터 공간 간의 맵, 커널과 이미지
- 직교성 및 내적: 내적 공간, 직교 기저, Gram-Schmidt
- 최소제곱해 및 직교분해: 데이터 근사와 해석의 도구
- 특이값분해(SVD): 차원 축소와 많은 응용의 기본 도구
- 응용 분야 예시: 컴퓨터 그래픽스, 데이터 분석, 머신러닝 등

원하는 학습 방향?
- 완전 초보에서 시작하고 싶나요?
- 특정 주제(예: 연립방정식 풀이, 고유값 문제, SVD 등)에 집중하고 싶나요?
- 이론 위주인가요, 아니면 문제 풀이 위주인가요?
- 예제 풀이를 직접 들려드리길 원하시나요? 아니면 간단한 개념 설명부터 시작할까요?

원하시면 바로 간단한 예제를 시작해 드리겠습니다. 예를 들어 2x2 연립방정식이나 벡터 공간의 기저 찾기 같은 문제를 같이 풀어볼 수 있어요. 어떤 방식이 좋으신가요?


#### 두 번째 대화 시작

In [38]:
# 두 번째 사용자와 대화
question2 = "안녕! 나는 자취생을 위한 요리가 궁금해."

response2 = chain_with_session_id.invoke(
    {"question": question2},
    config={"session_id": "user2"}  # 사용자2
)

print(f"사용자2: {question2}")
print(f"AI: {response2}")

사용자2: 안녕! 나는 자취생을 위한 요리가 궁금해.
AI: 좋아요! 자취생에게 helpful한 요리 아이디어를 정리해볼게요. 우선 몇 가지를 알려주시면 더 맞춤형으로 드릴 수 있어요:
- 현재 주방 기기(가스렌지/전기레인지, 전자레인지, 냄비/팬, 밥솥 등)
- 예산 수준(일주일 식비 대략 어느 정도?)
- 식단 선호/제약(한국식 vs 양식, 채식 여부, 알레르기)
- 요리에 투자 가능한 시간대(주중 하루에 20–30분 정도 가능? 1시간 정도 가능?)

바로 도움이 될 만한 기본 가이드와 간단 레시피를 몇 가지 드릴게요.

1) 자취생 필수 아이템(초보자용)
- 기본 양념: 간장, 된장, 고추장, 설탕, 소금, 다진 마늘, 참기름, 올리브유
- 주재료(다양하게 활용 가능한 것들): 쌀/밥, 달걀, 두부, 양파, 대파, 버섯, 김치, 참치캔, 토마토통조림, 파스타/면류
- 조리 도구: 냄비 하나와 프라이팬 하나로도 충분, 전자레인지 가능하면 더 편리
- 보관 팁: 남은 재료는 소분해 냉장고/냉동고에 나눠 보관, 냉동 보관 시 1–2주 이내 사용 추천

2) 바로 따라 할 수 있는 15–20분 레시피 5가지
- 김치볶음밥
  - 재료: 밥 1공기, 김치 1컵, 양파 1/4개, 대파 조금, 간장 1작은술, 참기름 약간
  - 방법: 팬에 기름 두르고 양파와 김치 볶다 밥 넣고 간장으로 간 맞추며 볶음. 다 완성 직전에 참기름 조금 뿌림. 원하면 계란후라이 올려도 좋아요.
  - 시간: 약 15분
- 참치김치찌개(가스레인지 한 냄비)
  - 재료: 김치 1컵, 참치캔 1개, 두부 1/2모, 물 2컵, 다진 마늘 1작은술, 대파
  - 방법: 냄비에 물•김치•마늘 끓이다가 참치와 두부 넣고 5분간 더 끓임. 소금/간장으로 간 맞추고 대파 올려 마무리.
  - 시간: 약 15–20분
- 두부버섯볶음
  - 재료: 두부 1/2모, 버섯 한 줌, 양파 1/4개, 간장 1–2큰술, 다진 마늘 1작은술
  - 방법: 팬에 기름 두르고 두부 노릇하게 굽다 버섯·양파 넣고 볶은

#### 각 대화를 구분할 수 있는지 검증

In [39]:
# 첫 번째 사용자와 대화
common_question = "제가 뭘 물어보려고 했었죠?"   # common_question 공통 질문

user1_response = chain_with_session_id.invoke(
    {"question": common_question},
    config={"session_id": "user1"}  # 사용자1
)

print(f"사용자1: {common_question}")
print(f"AI: {user1_response}")

# 두 번째 사용자와 대화
user2_response = chain_with_session_id.invoke(
    {"question": common_question},
    config={"session_id": "user2"}  # 사용자2
)

print(f"사용자2: {common_question}")
print(f"AI: {user2_response}")

사용자1: 제가 뭘 물어보려고 했었죠?
AI: 좋아요! 제가 추측하기로는, 앞으로 선형 대수를 어떤 방향으로 공부할지 결정하는 걸 도와달라는 의도였던 것 같아요.

아마 다음 중 하나를 물어보시려 했을 수 있습니다:
- 특정 주제에 집중: 예를 들어 연립방정식 풀이, 고유값/고유벡터, 역행렬, SVD 등
- 이론 vs 문제 풀이 비중 정하기
- 바로 예제 풀이로 들어가기

원하시는 방향을 골라주시면 바로 시작하겠습니다. 아래 중 하나를 선택해 주세요, 아니면 다른 주제도 말씀해 주세요.
- 2x2 연립방정식 풀이 예제 같이 풀기
- 행렬의 역행행렬 구하기와 관련 문제
- 고유값/고유벡터 구하기와 대각화
- Gram-Schmidt로 직교 기저 만들기
- 최소제곱해 구하기 또는 SVD의 직관 설명 및 예제

또는 “먼저 이론 설명부터 시작하자”처럼 느슨한 방향도 좋습니다. 어떤 방식이 좋으신가요? 바로 시작해 드리겠습니다.
사용자2: 제가 뭘 물어보려고 했었죠?
AI: 좋아요! 정확히 무엇을 물어보려 하셨는지 기억이 안 나면 제가 몇 가지 흔한 방향을 추려볼게요. 아래 중 하나를 골라 주시면 바로 상세하게 도와드릴게요.

- 1) 1주 식단표(재료비 포함)와 주간 장보기 리스트
- 2) 현재 보유 주방기기로 맞춘 초간단 레시피 5~7가지
- 3) 특정 재료 하나로 만드는 빠른 레시피 아이디어 모음
- 4) 예산에 맞춘 식비 관리 팁과 주별 레시피 루틴
- 5) 특정 식단 선호/제약(한국식 vs 양식, 채식, 알레르기 등)을 반영한 맞춤 레시피

원하시는 방향을 골라 주세요. 또는 기억나지 않는다면, 바로 맞춤형 1주 식단표를 만들어 드려도 될까요? 또한 현재 가진 재료나 기기, 주당 가능 시간, 선호 음식, 알레르기 여부를 알려주시면 더 정확히 맞춰 드릴게요.


#### 참고. 사용자별 결과 출력

In [40]:
for key, value in user_chats.items(): # items() : key, value값을 꺼내옴
    for message in value.messages:
        speaker = key if "human" in str(type(message)) else "AI"
        print(f"{speaker}: {message.content}")
    print()

user1: 안녕하세요. 저는 선형 대수가 궁금합니다.
AI: 반갑습니다! 선형 대수는 벡터와 행렬을 다루며, 선형 관계와 변환의 구조를 이해하는 학문입니다. 시작하기에 앞서 몇 가지 핵심 주제를 간단히 소개하고, 원하시는 방향으로 맞춰 드릴게요.

주요 개념 정리
- 벡터와 벡터 공간: 벡터의 합성, 스칼라 곱, 기저와 차원
- 행렬과 연산: 덧셈/스칼라곱, 행렬 곱, 역행렬, 행렬식
- 선형 시스템: 연립선형방정식의 해(해의 존재성과 유일성), 랭크
- 고유값/고유벡터와 대각화: 선형 변환의 구조를 단순화
- 선형 변환과 기하적 해석: 벡터 공간 간의 맵, 커널과 이미지
- 직교성 및 내적: 내적 공간, 직교 기저, Gram-Schmidt
- 최소제곱해 및 직교분해: 데이터 근사와 해석의 도구
- 특이값분해(SVD): 차원 축소와 많은 응용의 기본 도구
- 응용 분야 예시: 컴퓨터 그래픽스, 데이터 분석, 머신러닝 등

원하는 학습 방향?
- 완전 초보에서 시작하고 싶나요?
- 특정 주제(예: 연립방정식 풀이, 고유값 문제, SVD 등)에 집중하고 싶나요?
- 이론 위주인가요, 아니면 문제 풀이 위주인가요?
- 예제 풀이를 직접 들려드리길 원하시나요? 아니면 간단한 개념 설명부터 시작할까요?

원하시면 바로 간단한 예제를 시작해 드리겠습니다. 예를 들어 2x2 연립방정식이나 벡터 공간의 기저 찾기 같은 문제를 같이 풀어볼 수 있어요. 어떤 방식이 좋으신가요?
user1: 제가 뭘 물어보려고 했었죠?
AI: 좋아요! 제가 추측하기로는, 앞으로 선형 대수를 어떤 방향으로 공부할지 결정하는 걸 도와달라는 의도였던 것 같아요.

아마 다음 중 하나를 물어보시려 했을 수 있습니다:
- 특정 주제에 집중: 예를 들어 연립방정식 풀이, 고유값/고유벡터, 역행렬, SVD 등
- 이론 vs 문제 풀이 비중 정하기
- 바로 예제 풀이로 들어가기

원하시는 방향을 골라주시면 바로 시작하겠습니다. 아래 중 하나를 선택해 주세요, 아니면 다른 주제도 말씀해 주세요.
- 2x2 연